In [2]:
# CÉLULA 1 — Carga e preparo para regressões

import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

BASE_DIR = r"C:\Users\Usuario\Downloads\Bruno_USP\Documentos\Censo_Religião"
caminho_dados = os.path.join(BASE_DIR, "data", "processed", "censo_religiao_renda.csv")

df = pd.read_csv(caminho_dados)

# Recria as variáveis derivadas (mesmas do notebook 03)
def definir_regiao(uf):
    primeiro = str(uf)[0]
    if primeiro == "1":
        return "Norte"
    elif primeiro == "2":
        return "Nordeste"
    elif primeiro == "3":
        return "Sudeste"
    elif primeiro == "4":
        return "Sul"
    elif primeiro == "5":
        return "Centro-Oeste"
    return "Indefinida"

df["regiao"] = df["uf"].apply(definir_regiao)
df["sexo_label"] = df["sexo"].map({1: "Masculino", 2: "Feminino"})
df["raca_label"] = df["raca"].map({1: "Branca", 2: "Preta", 3: "Amarela", 4: "Parda", 5: "Indígena"})

def tem_superior(linha):
    if linha["ano"] in [1991, 2000]:
        return 15 <= linha["escolaridade_original"] <= 17
    elif linha["ano"] == 2010:
        return linha["escolaridade_original"] == 4
    return False

grupos_foco = ["Católica", "Evangélica Tradicional", "Evangélica Pentecostal",
               "Sem religião", "Mediúnica/Espírita"]

# Base de regressão: renda positiva, idade produtiva, grupos foco
base_reg = df[(df["idade"] >= 18) & (df["idade"] <= 65) &
              (df["renda_sm"] > 0) & (df["renda_sm"] <= 200) &
              (df["grupo_religioso"].isin(grupos_foco))].copy()

base_reg["superior"] = base_reg.apply(tem_superior, axis=1)
base_reg["log_renda"] = np.log(base_reg["renda_sm"])
base_reg["mulher"] = (base_reg["sexo"] == 2).astype(int)

print(f"Base de regressão: {len(base_reg)} registros")
print(base_reg["grupo_religioso"].value_counts())

Base de regressão: 144046 registros
grupo_religioso
Católica                  112856
Evangélica Pentecostal     12840
Sem religião                9127
Evangélica Tradicional      6982
Mediúnica/Espírita          2241
Name: count, dtype: int64


In [3]:
# CÉLULA 2 — Regressão OLS: log(renda) ~ religião + controles

# Categoria de referência: Católica (maior grupo)
base_reg["grupo_religioso"] = pd.Categorical(
    base_reg["grupo_religioso"],
    categories=["Católica", "Evangélica Tradicional", "Evangélica Pentecostal",
                "Sem religião", "Mediúnica/Espírita"]
)

modelo = smf.ols(
    "log_renda ~ C(grupo_religioso) + idade + I(idade**2) + mulher + "
    "C(raca_label) + superior + C(regiao) + C(ano)",
    data=base_reg
).fit()

print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.397
Model:                            OLS   Adj. R-squared:                  0.397
Method:                 Least Squares   F-statistic:                     5254.
Date:                Fri, 12 Jun 2026   Prob (F-statistic):               0.00
Time:                        13:52:44   Log-Likelihood:            -1.8880e+05
No. Observations:              143738   AIC:                         3.776e+05
Df Residuals:                  143719   BIC:                         3.778e+05
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

In [4]:
# CÉLULA 3 — Logística: probabilidade de diploma na geração de berço (1976-86)

# Base: geração em 2010, SEM filtro de renda (educação não exige estar empregado)
base_dip = df[(df["ano"] == 2010) &
              (df["idade"] >= 24) & (df["idade"] <= 34) &
              (df["grupo_religioso"].isin(grupos_foco))].copy()

base_dip["superior"] = base_dip.apply(tem_superior, axis=1).astype(int)
base_dip["mulher"] = (base_dip["sexo"] == 2).astype(int)

base_dip["grupo_religioso"] = pd.Categorical(
    base_dip["grupo_religioso"],
    categories=["Católica", "Evangélica Tradicional", "Evangélica Pentecostal",
                "Sem religião", "Mediúnica/Espírita"]
)

modelo_logit = smf.logit(
    "superior ~ C(grupo_religioso) + idade + mulher + C(raca_label) + C(regiao)",
    data=base_dip
).fit()

print(modelo_logit.summary())

# Razões de chance (odds ratios) — mais fáceis de interpretar
print("\n===== RAZÕES DE CHANCE (odds ratios) =====")
odds = np.exp(modelo_logit.params)
conf = np.exp(modelo_logit.conf_int())
tabela_odds = pd.DataFrame({"odds_ratio": odds.round(3),
                            "IC_2.5%": conf[0].round(3),
                            "IC_97.5%": conf[1].round(3)})
print(tabela_odds.loc[[i for i in tabela_odds.index if "grupo_religioso" in i]])

Optimization terminated successfully.
         Current function value: 0.268079
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:               superior   No. Observations:                23478
Model:                          Logit   Df Residuals:                    23463
Method:                           MLE   Df Model:                           14
Date:                Fri, 12 Jun 2026   Pseudo R-squ.:                  0.1601
Time:                        14:46:04   Log-Likelihood:                -6294.0
converged:                       True   LL-Null:                       -7493.4
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
Intercept                               

In [5]:
# CÉLULA 4 — Qui-quadrado: associação religião × diploma (geração de berço)

from scipy.stats import chi2_contingency

tabela_contingencia = pd.crosstab(base_dip["grupo_religioso"], base_dip["superior"])
print("Tabela de contingência (religião × diploma):")
print(tabela_contingencia)

chi2, p_valor, gl, esperado = chi2_contingency(tabela_contingencia)

print(f"\nQui-quadrado: {chi2:.1f}")
print(f"Graus de liberdade: {gl}")
print(f"p-valor: {p_valor:.2e}")

if p_valor < 0.05:
    print("\nConclusão: a associação entre religião e diploma é estatisticamente significante.")
else:
    print("\nConclusão: não há evidência de associação significante.")

Tabela de contingência (religião × diploma):
superior                    0     1
grupo_religioso                    
Católica                15161  1532
Evangélica Tradicional    956   189
Evangélica Pentecostal   3097   149
Sem religião             1841   270
Mediúnica/Espírita        138   145

Qui-quadrado: 740.7
Graus de liberdade: 4
p-valor: 5.29e-159

Conclusão: a associação entre religião e diploma é estatisticamente significante.


In [6]:
# CÉLULA 5 — Tabela-resumo dos principais resultados

print("="*70)
print("TABELA 1 — Efeito da religião sobre a renda (OLS, log da renda em SM)")
print("Referência: Católica | Controles: idade, sexo, raça, diploma, região, ano")
print("="*70)

coefs_ols = modelo.params
pvals_ols = modelo.pvalues
for nome in coefs_ols.index:
    if "grupo_religioso" in nome:
        religiao = nome.split("T.")[1].rstrip("]")
        efeito_pct = (np.exp(coefs_ols[nome]) - 1) * 100
        sig = "***" if pvals_ols[nome] < 0.001 else "**" if pvals_ols[nome] < 0.01 else "*" if pvals_ols[nome] < 0.05 else ""
        print(f"{religiao:30s} {coefs_ols[nome]:+.3f}  ({efeito_pct:+.1f}%) {sig}")

print()
print("="*70)
print("TABELA 2 — Religião de berço e chance de diploma (Logística, odds ratio)")
print("Geração 1976-86 em 2010 | Referência: Católica | Controles: idade, sexo, raça, região")
print("="*70)

coefs_log = modelo_logit.params
pvals_log = modelo_logit.pvalues
for nome in coefs_log.index:
    if "grupo_religioso" in nome:
        religiao = nome.split("T.")[1].rstrip("]")
        odds = np.exp(coefs_log[nome])
        sig = "***" if pvals_log[nome] < 0.001 else "**" if pvals_log[nome] < 0.01 else "*" if pvals_log[nome] < 0.05 else ""
        print(f"{religiao:30s} OR = {odds:.2f} {sig}")

print("\nSignificância: *** p<0,001 | ** p<0,01 | * p<0,05")

TABELA 1 — Efeito da religião sobre a renda (OLS, log da renda em SM)
Referência: Católica | Controles: idade, sexo, raça, diploma, região, ano
Evangélica Tradicional         +0.083  (+8.7%) ***
Evangélica Pentecostal         -0.032  (-3.2%) ***
Sem religião                   +0.100  (+10.5%) ***
Mediúnica/Espírita             +0.502  (+65.2%) ***

TABELA 2 — Religião de berço e chance de diploma (Logística, odds ratio)
Geração 1976-86 em 2010 | Referência: Católica | Controles: idade, sexo, raça, região
Evangélica Tradicional         OR = 1.70 ***
Evangélica Pentecostal         OR = 0.45 ***
Sem religião                   OR = 1.28 **
Mediúnica/Espírita             OR = 4.78 ***

Significância: *** p<0,001 | ** p<0,01 | * p<0,05
